# Hardware Trojans in AES-128: Starter Notebook

This notebook sets up everything you need for the assignment:

1. installs Icarus Verilog and a waveform reader,
2. pulls the Verilog files from the assignment folder into **one working folder per design**,
3. compiles and runs the provided testbench for each design and plots the waveforms,
4. gives you a testbench template to start Part C.

| Tag | Design | Original folder |
|-----|--------|-----------------|
| `AES` | Trojan-free reference | `Main_Trojans/AES/` |
| `T1` | Functional Trojan | `Main_Trojans/Functional_Trojan/` |
| `T2` | Timing-dependent functional Trojan | `Main_Trojans/Functional_Timing_Trojan/` |
| `T3` | Degradation Trojan | `Main_Trojans/Degradation_attack_Trojan/` |

You may use certain sections of this notebook, none, or all of it as per how you want to approach this assignment.

## 1. Install the tools

Installs Icarus Verilog (`iverilog`, `vvp`) and the `vcdvcd` Python package used to read waveform (`.vcd`) files.

In [ ]:
import sys, shutil, subprocess

IN_COLAB = "google.colab" in sys.modules

if shutil.which("iverilog") is None:
    if IN_COLAB:
        subprocess.run("apt-get -qq update && apt-get -qq install -y iverilog", shell=True, check=True)
    else:
        raise RuntimeError("Icarus Verilog was not found. Install it first "
                           "(Ubuntu: sudo apt install iverilog, macOS: brew install icarus-verilog) "
                           "and re-run this cell.")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vcdvcd"], check=True)
print(subprocess.run(["iverilog", "-V"], capture_output=True, text=True).stdout.splitlines()[0])

## 2. Get the assignment files

Choose where the assignment comes from by setting `SOURCE`:

- `"upload"`: (Colab) you will be asked to upload the assignment `.zip` file.
- `"drive"`: (Colab) the assignment folder or `.zip` is in your Google Drive; set `DRIVE_PATH`.
- `"local"`: (local Jupyter, or a file you already placed in Colab) set `LOCAL_PATH` to the folder or `.zip`.

I haven't exhaustively tested this exhaustively, so you may do something similar to what was done in assignment 2 if you have trouble with this, or any other approach.

In [ ]:
SOURCE     = "upload"                                   # "upload", "drive" or "local"
DRIVE_PATH = "/content/drive/MyDrive/Hardware_Trojan"   # folder or .zip in your Google Drive
LOCAL_PATH = "Hardware_Trojan"                          # folder or .zip on this machine

In [ ]:
import zipfile
from pathlib import Path

SRC_DIR = Path("assignment_src")

def _unzip(zip_path):
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(SRC_DIR)
    return SRC_DIR

if SOURCE == "upload":
    from google.colab import files
    uploaded = files.upload()                      # pick the assignment .zip
    zips = [name for name in uploaded if name.lower().endswith(".zip")]
    if not zips:
        raise ValueError("Please upload the assignment .zip file.")
    root = _unzip(zips[0])
elif SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    p = Path(DRIVE_PATH)
    root = _unzip(p) if p.suffix.lower() == ".zip" else p
elif SOURCE == "local":
    p = Path(LOCAL_PATH)
    root = _unzip(p) if p.suffix.lower() == ".zip" else p
else:
    raise ValueError('SOURCE must be "upload", "drive" or "local".')

matches = [d for d in Path(root).rglob("Main_Trojans")
           if d.is_dir() and "__MACOSX" not in d.parts]
if not matches:
    raise FileNotFoundError(f"Could not find a 'Main_Trojans' folder inside {root}.")
MAIN = matches[0]
print("Using design files from:", MAIN.resolve())

## 3. Pull the Verilog files into separate working folders

Each design gets its own folder under `designs/` containing its three RTL files (`aes_128.v`, `round.v`, `table.v`) and its testbench (`test_aes_128.v`). The provided testbenches already dump a waveform to `wave.vcd` and end with `$finish`.

You work on these copies, so the original assignment files are never modified. Re-running this cell does **not** overwrite folders that already exist. Set `OVERWRITE = True` to restore a fresh copy (this deletes your files in `designs/`).

In [ ]:
DESIGNS = {
    "AES": ("AES",                       "Trojan-free reference"),
    "T1":  ("Functional_Trojan",         "Functional Trojan"),
    "T2":  ("Functional_Timing_Trojan",  "Timing-dependent functional Trojan"),
    "T3":  ("Degradation_attack_Trojan", "Degradation Trojan"),
}
RTL_FILES = ["aes_128.v", "round.v", "table.v"]
TB_FILE   = "test_aes_128.v"
WORK      = Path("designs")
OVERWRITE = False

for tag, (folder, desc) in DESIGNS.items():
    src, dst = MAIN / folder, WORK / tag
    if dst.exists() and OVERWRITE:
        shutil.rmtree(dst)
    if not dst.exists():
        dst.mkdir(parents=True)
        for name in RTL_FILES + [TB_FILE]:
            shutil.copy(src / name, dst / name)
    print(f"{tag:4s} {desc:38s} -> {dst}/  ({', '.join(sorted(p.name for p in dst.glob('*.v')))})")

## 4. Helper functions

Run this cell once. It defines:

- `simulate(design, tb=...)`: compiles the testbench together with that design's RTL in `designs/<design>/` and runs it. The testbench is compiled **first**, so the RTL files inherit its `` `timescale 1ns / 1ps `` (the RTL files have no `timescale` of their own).
- `list_signals(design, pattern)`: lists the signals recorded in the waveform file.
- `changes(design, signal, start_ns, end_ns)`: prints every value change of a signal, in hex.
- `plot_waves(design, signals, start_ns, end_ns)`: draws a waveform diagram.

Signals can be given by short name (`"out"`, `"uut.s0"`) or full name (`"test_aes_128.out[127:0]"`). Times are in ns.

Again, the testing on this was somewhat limited, so if you have trouble with this, you may use the approach that was used in assignment 2.

In [ ]:
import math
import matplotlib.pyplot as plt
from vcdvcd import VCDVCD

def simulate(design, tb=TB_FILE, extra_files=(), show_log=True, timeout=900):
    """Compile tb + extra_files + the design's RTL in designs/<design>/ and run the simulation."""
    d = WORK / design
    sources = [tb, *extra_files, *RTL_FILES]          # testbench first: RTL inherits its `timescale
    comp = subprocess.run(["iverilog", "-g2012", "-o", "sim", *sources],
                          cwd=d, capture_output=True, text=True)
    if comp.returncode != 0:
        print(comp.stdout + comp.stderr)
        raise RuntimeError(f"Compilation failed for {design}")
    if comp.stderr.strip():
        print(comp.stderr)
    try:
        run = subprocess.run(["vvp", "-n", "sim"], cwd=d, capture_output=True, text=True, timeout=timeout)
    except subprocess.TimeoutExpired:
        raise RuntimeError(f"Simulation of {design} did not finish within {timeout} s. "
                           "Does your testbench end with $finish?")
    log = run.stdout + run.stderr
    if show_log:
        print(f"--- {design}: {tb} ---\n{log}")
    return log

_vcd_cache = {}
def _load(design, vcd):
    path = WORK / design / vcd
    key = (str(path), path.stat().st_mtime)
    if key not in _vcd_cache:
        _vcd_cache[key] = VCDVCD(str(path))
    return _vcd_cache[key]

def _to_ns(v):
    return float(v.get_timescale()["factor"]) * 1e9   # VCD time unit expressed in ns

def _resolve(v, name):
    if name in v.signals:
        return name
    base = lambda s: s.split("[")[0]
    hits = [s for s in v.signals if base(s) == name or base(s).endswith("." + name)]
    if not hits:
        raise KeyError(f"No signal matching '{name}'. Use list_signals() to see what is recorded.")
    return min(hits, key=lambda s: s.count("."))       # prefer the highest level of hierarchy

def _full(bits, width):
    """Extend a VCD value to its full width (Verilog rules: x/z extend, otherwise 0)."""
    pad = bits[0] if bits[0] in "xXzZ" else "0"
    return bits.rjust(width, pad) if len(bits) < width else bits

def _hex(bits, width):
    bits = _full(bits, width)
    bits = bits.rjust(4 * math.ceil(width / 4), "0")
    out = []
    for i in range(0, len(bits), 4):
        nib = bits[i:i+4].lower()
        out.append("x" if "x" in nib else "z" if "z" in nib else format(int(nib, 2), "x"))
    return "".join(out)

def list_signals(design, pattern="", vcd="wave.vcd"):
    for s in _load(design, vcd).signals:
        if pattern in s:
            print(s)

def changes(design, signal, start_ns=0, end_ns=None, vcd="wave.vcd"):
    v = _load(design, vcd); name = _resolve(v, signal); sig = v[name]
    width, f = int(sig.size), _to_ns(v)
    print(f"{name}  ({width} bits)")
    for t, val in sig.tv:
        t_ns = t * f
        if t_ns < start_ns or (end_ns is not None and t_ns > end_ns):
            continue
        print(f"{t_ns:>14,.1f} ns   {_hex(val, width)}")

def plot_waves(design, signals=("clk", "state", "key", "out"), start_ns=None, end_ns=None,
               vcd="wave.vcd", title=None):
    v = _load(design, vcd); f = _to_ns(v)
    t0 = 0.0 if start_ns is None else start_ns
    t1 = v.get_endtime() * f if end_ns is None else end_ns
    fig, axes = plt.subplots(len(signals), 1, sharex=True,
                             figsize=(14, 0.9 * len(signals) + 0.8), squeeze=False)
    for ax, s in zip(axes[:, 0], signals):
        name = _resolve(v, s); sig = v[name]; width = int(sig.size)
        tv = [(t * f, val) for t, val in sig.tv]
        # value at t0, then the changes inside the window
        cur = [val for t, val in tv if t <= t0]
        segs = [(t0, cur[-1] if cur else "x")] + [(t, val) for t, val in tv if t0 < t < t1]
        bounds = [t for t, _ in segs[1:]] + [t1]
        ax.set_yticks([]); ax.set_ylim(-0.2, 1.2)
        ax.set_ylabel(name.split(".", 1)[-1], rotation=0, ha="right", va="center", fontsize=9)
        for spine in ("top", "right", "left"):
            ax.spines[spine].set_visible(False)
        if width == 1:
            xs, ys = [], []
            for (ts, val), te in zip(segs, bounds):
                y = 1 if val == "1" else 0 if val == "0" else 0.5
                xs += [ts, te]; ys += [y, y]
            ax.plot(xs, ys, color="tab:blue", lw=1)
        else:
            span = t1 - t0
            for (ts, val), te in zip(segs, bounds):
                full = _hex(val, width)
                unknown = all(ch in "xz" for ch in full)
                e = min(span * 0.004, (te - ts) / 3)
                ax.fill([ts, ts + e, te - e, te, te - e, ts + e], [0.5, 1, 1, 0.5, 0, 0],
                        facecolor="tab:red" if unknown else "white", alpha=0.3 if unknown else 1,
                        edgecolor="tab:red" if unknown else "tab:blue", lw=1)
                frac = (te - ts) / span
                if frac > 0.03:
                    max_chars = max(4, int(frac * 160))
                    label = "x" if unknown else full if len(full) <= max_chars else \
                            full[:max_chars // 2 - 1] + "…" + full[-(max_chars // 2 - 1):]
                    ax.text((ts + te) / 2, 0.5, label, ha="center", va="center",
                            fontsize=8, family="monospace")
    axes[-1, 0].set_xlim(t0, t1)
    axes[-1, 0].set_xlabel("time (ns)")
    fig.suptitle(title or f"{design}: {vcd}", fontsize=11)
    fig.tight_layout()
    plt.show()

print("Helpers ready.")

## 5. Run the provided testbench on every design

The same testbench (`test_aes_128.v`) is in all four folders. It applies the key `000102030405060708090A0B0C0D0E0F` and five plaintexts, one per clock cycle, and records a waveform. It does not check anything.

To look inside a design, list its internal signals (for example `list_signals("T1", "uut")`) and add them to `plot_waves`, e.g. `signals=("clk", "state", "out", "uut.s0")`. The provided testbench records the testbench signals and the signals declared directly inside `aes_128`. Change `$dumpvars(2, ...)` to `$dumpvars(0, ...)` in a testbench to record every submodule.

In [ ]:
for tag in DESIGNS:
    simulate(tag)
    plot_waves(tag, title=f"{tag}: provided testbench")

In [ ]:
for tag in DESIGNS:
    changes(tag, "out")
    print()

## 6. Your own testbench (Part C)

Start from the template below. It already contains the clock, the design instance, a cycle counter, the waveform dump (`my_wave.vcd`) and `$finish`. Add your stimulus and self-checking logic where the `TODO` comments are.

The cell writes the same testbench into every folder listed in `RUN_ON`, so you can run it on the Trojan-free design and on a Trojan design and compare (Part C requires that your test reports `PASS` on `AES`).

Tips:
- Icarus Verilog supports only part of SystemVerilog. Stick to modules, tasks, functions, `$urandom`, and plain `if`/`$display` checks.
- Change inputs on the falling edge of the clock (as the template does) so they are stable at the rising edge.
- If you instantiate the Trojan-free design as a reference model next to a Trojan design, copy its files under new module names (for example, prefix every module with `ref_`) so the names do not collide.

In [ ]:
MY_TB_NAME = "my_tb.v"
RUN_ON     = ["AES", "T1"]          # designs to run your testbench on

MY_TB = r"""
`timescale 1ns / 1ps

module my_tb;
    reg          clk   = 0;
    reg  [127:0] state = 0;
    reg  [127:0] key   = 0;
    wire [127:0] out;

    aes_128 uut (.clk(clk), .state(state), .key(key), .out(out));

    always #5 clk = ~clk;                      // 10 ns clock

    integer cycle = 0;                         // counts rising edges
    always @(posedge clk) cycle <= cycle + 1;

    initial begin
        $dumpfile("my_wave.vcd");
        $dumpvars(2, my_tb);

        // TODO: apply your stimulus. Change inputs on the falling edge, e.g.
        @(negedge clk);
        state = 128'h0;
        key   = 128'h0;

        $finish;
    end
endmodule
"""

for tag in RUN_ON:
    (WORK / tag / MY_TB_NAME).write_text(MY_TB.lstrip())
for tag in RUN_ON:
    simulate(tag, tb=MY_TB_NAME)
    plot_waves(tag, vcd="my_wave.vcd", title=f"{tag}: {MY_TB_NAME}")